# FastSpeech 2: English and Telugu TTS

**Reference paper:** [FastSpeech 2: Fast and High-Quality End-to-End Text to Speech](https://arxiv.org/abs/2006.04558)  
**Repository:** [abhiram-k-dev/fastspeech2-english-telugu-tts](https://github.com/abhiram-k-dev/fastspeech2-english-telugu-tts)

This notebook runs the complete English and Telugu TTS pipeline:
1. Environment setup and dependency installation
2. Mount Google Drive (for datasets and checkpoints)
3. Clone/update the repository
4. English preprocessing + training + inference
5. Telugu preprocessing + training + inference
6. Display generated audio and variance adaptor plots

**Requirements:** T4 GPU runtime, Google Drive with datasets pre-downloaded.

## Cell 1: Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Expected structure on Google Drive:
# MyDrive/tts_assignment/
#   raw/english/raw_en/<speaker>/<utt>.wav + .lab
#   raw/telugu/raw_te/<speaker>/<utt>.wav + .lab
#   processed/english/aligned_en/<speaker>/<utt>.TextGrid
#   processed/english/en_merged.dict
#   processed/english/manifest_clean.csv
#   processed/telugu/aligned_te/<speaker>/<utt>.TextGrid
#   processed/telugu/manifest_clean.csv
#   checkpoints/english/
#   checkpoints/telugu/
#   results/

## Cell 2: Install Dependencies

In [ ]:
# Install Python packages
!pip install -q g2p-en inflect tgt pyworld librosa soundfile

# Install Telugu font for readable plot labels
!apt-get install -q -y fonts-telu

print('Dependencies installed.')

## Cell 3: Clone Repository and Set Up Paths

In [ ]:
import os

REPO_DIR = '/content/fastspeech2-english-telugu-tts'
GDRIVE_ROOT = '/content/drive/MyDrive/tts_assignment'

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/abhiram-k-dev/fastspeech2-english-telugu-tts.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

os.chdir(REPO_DIR)
print('Working directory:', os.getcwd())

## Cell 4: Apply Compatibility Patches

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)
import tts_helpers as h
h.patch_repo(REPO_DIR)
print('Patches applied.')

## Cell 5: Download HiFi-GAN Vocoder

In [ ]:
import os

hifigan_dir = f'{REPO_DIR}/hifigan'
vocoder_path = f'{hifigan_dir}/generator_universal.pth.tar'

if not os.path.exists(vocoder_path):
    # Download from the official HiFi-GAN release
    !gdown --id 1-eEYTB5Av9jNql0WGBlRoi-WH2J7bp5Y -O {vocoder_path}
    print('HiFi-GAN universal vocoder downloaded.')
else:
    print('HiFi-GAN vocoder already present.')

---
# ENGLISH PIPELINE

## Cell 6: English Preprocessing

In [ ]:
os.chdir(REPO_DIR)
!python English/preprocess.py English/config.yaml
!python make_configs.py English/config.yaml

## Cell 7: English Training (10,000 steps)

In [ ]:
os.chdir(REPO_DIR)
# Change --restore_step if resuming from a checkpoint
!python train.py --restore_step 0 \
    -p config/preprocess.yaml \
    -m config/model.yaml \
    -t config/train.yaml

## Cell 8: English Inference and Variance Plots

In [ ]:
os.chdir(REPO_DIR)
!python English/inference.py --restore_step 10000 \
    -p config/preprocess.yaml \
    -m config/model.yaml \
    -t config/train.yaml \
    --out Results/English

## Cell 9: Listen to English Generated Audio

In [ ]:
import IPython.display as ipd
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

wav_dir = f'{REPO_DIR}/Results/English/test_generated_audios'
for wav in sorted(glob.glob(f'{wav_dir}/*.wav')):
    name = os.path.basename(wav).replace('.wav', '')
    print(f'\n--- {name} ---')
    display(ipd.Audio(wav))

# Show variance plots
plot_dir = f'{REPO_DIR}/Results/English/variance_adaptor_plots'
for feature in ('duration', 'pitch', 'energy'):
    pngs = sorted(glob.glob(f'{plot_dir}/{feature}/*.png'))
    if pngs:
        fig, axes = plt.subplots(1, len(pngs), figsize=(4*len(pngs), 3))
        if len(pngs) == 1:
            axes = [axes]
        for ax, png in zip(axes, pngs):
            ax.imshow(mpimg.imread(png))
            ax.axis('off')
            ax.set_title(os.path.basename(png).replace('.png', ''), fontsize=8)
        fig.suptitle(f'English — {feature}', fontsize=10)
        plt.tight_layout()
        plt.show()

---
# TELUGU PIPELINE

## Cell 10: Telugu Preprocessing (also rebuilds text/symbols.py)

In [ ]:
os.chdir(REPO_DIR)
!python Telugu/preprocess.py Telugu/config.yaml
!python make_configs.py Telugu/config.yaml

# Verify Telugu symbols were built
import json
syms = json.load(open(f'{REPO_DIR}/text/telugu_symbols.json', encoding='utf-8'))
print(f'Telugu grapheme symbols: {len(syms)}')
print(syms)

## Cell 11: Telugu Training (10,000 steps)

In [ ]:
os.chdir(REPO_DIR)
# Change --restore_step if resuming from a checkpoint
!python train.py --restore_step 0 \
    -p config/preprocess_telugu.yaml \
    -m config/model_telugu.yaml \
    -t config/train.yaml

## Cell 12: Telugu Inference and Variance Plots

In [ ]:
os.chdir(REPO_DIR)
!python Telugu/inference.py --restore_step 10000 \
    -p config/preprocess_telugu.yaml \
    -m config/model_telugu.yaml \
    -t config/train.yaml \
    --out Results/Telugu

## Cell 13: Listen to Telugu Generated Audio

In [ ]:
import IPython.display as ipd
import glob
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

wav_dir = f'{REPO_DIR}/Results/Telugu/test_generated_audios'
for wav in sorted(glob.glob(f'{wav_dir}/*.wav')):
    name = os.path.basename(wav).replace('.wav', '')
    print(f'\n--- {name} ---')
    display(ipd.Audio(wav))

# Show variance plots
plot_dir = f'{REPO_DIR}/Results/Telugu/variance_adaptor_plots'
for feature in ('duration', 'pitch', 'energy'):
    pngs = sorted(glob.glob(f'{plot_dir}/{feature}/*.png'))
    if pngs:
        fig, axes = plt.subplots(1, len(pngs), figsize=(4*len(pngs), 3))
        if len(pngs) == 1:
            axes = [axes]
        for ax, png in zip(axes, pngs):
            ax.imshow(mpimg.imread(png))
            ax.axis('off')
            ax.set_title(os.path.basename(png).replace('.png', ''), fontsize=8)
        fig.suptitle(f'Telugu — {feature}', fontsize=10)
        plt.tight_layout()
        plt.show()